# Fraud detection for transaction review
**Diogo Bertozi | Artefact data science case**

The model prioritizes transactions for investigation. Missed fraud can expose the institution to loss. False alerts use analyst time and can interrupt legitimate customers. Review capacity and error costs were unspecified, so the operating threshold is provisional.

**Run:** Start Jupyter in the folder containing both original CSVs and run all cells with Python 3 and pandas, NumPy, scikit-learn and Matplotlib installed. This creates the classified CSV in `submission/` and supporting figures and metrics in `_build/`. The CSV keeps original column names for compatibility.

In [ ]:
import os
for variable in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[variable] = '4'

import json
import platform
from pathlib import Path
from time import perf_counter

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Image, Markdown, display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

started = perf_counter()
versions = dict(python=platform.python_version(), pandas=pd.__version__, numpy=np.__version__,
                sklearn=sklearn.__version__, matplotlib=matplotlib.__version__)
charts = Path('_build/charts')
charts.mkdir(parents=True, exist_ok=True)
Path('submission').mkdir(exist_ok=True)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.labelcolor': '#16324a', 'text.color': '#16324a',
                     'axes.titlecolor': '#16324a', 'font.size': 11})
teal = '#007f78'

def show_chart(fig, filename):
    fig.tight_layout()
    fig.savefig(charts / filename, dpi=160, bbox_inches='tight', facecolor='white')
    display(Image(filename=str(charts / filename)))
    plt.close(fig)

print(versions)

## 1. Data audit and chronological split
The target is the supplied historical fraud label. Label completeness, maturity and source-system definitions could not be independently confirmed. A missing minute represents an absent transaction record, so no rows are fabricated.

January–September fits the models. October–November selects the model and threshold. December measures performance of those frozen choices. Initial exploratory profiling inspected December labels, so December is a temporal evaluation set with that limitation. January has no labels.

In [ ]:
history_path = Path('transacoes_historico_2025.csv')
january_path = Path('transacoes_jan2026.csv')
target = 'flag_fraude'
timestamp = 'datahora_transacao'
history = pd.read_csv(history_path, dtype={'codigo_postal': str})
january = pd.read_csv(january_path, dtype={'codigo_postal': str})
assert history.columns.drop(target).tolist() == january.columns.tolist()
assert len(history) == 523_525 and len(january) == 44_451
assert set(history[target].unique()) == {False, True} and pd.api.types.is_bool_dtype(history[target])

times = {}
audit = []
for name, frame in [('2025 history', history), ('January 2026', january)]:
    dates = pd.to_datetime(frame[timestamp], format='%Y-%m-%d %H:%M:%S', errors='raise')
    missing = int(frame.isna().sum().sum())
    duplicates = int(frame.duplicated().sum())
    duplicate_times = int(dates.duplicated().sum())
    assert missing == duplicates == duplicate_times == 0, name
    assert pd.api.types.is_bool_dtype(frame['is_new_device'])
    assert np.isfinite(frame['valor_transacao']).all() and frame['valor_transacao'].ge(0).all()
    times[name] = dates
    audit.append(dict(dataset=name, rows=len(frame), missing_cells=missing,
                      duplicate_rows=duplicates, duplicate_timestamps=duplicate_times))
display(pd.DataFrame(audit))

dates, january_dates = times.values()
assert dates.dt.year.eq(2025).all() and january_dates.dt.to_period('M').eq(pd.Period('2026-01')).all()
train = dates < '2025-10-01'
validation = (dates >= '2025-10-01') & (dates < '2025-12-01')
test = dates >= '2025-12-01'
y = history[target].astype(int)
assert (train.astype(int) + validation.astype(int) + test.astype(int)).eq(1).all()
assert dates[train].max() < dates[validation].min() <= dates[validation].max() < dates[test].min()
assert dates[test].max() < january_dates.min()
splits = {
    name: {'rows': int(mask.sum()), 'fraud_count': int(y[mask].sum()),
           'fraud_rate': float(y[mask].mean())}
    for name, mask in [('train', train), ('validation', validation), ('test', test)]
}
assert [v['rows'] for v in splits.values()] == [391_566, 87_508, 44_451]
assert all(y[mask].nunique() == 2 for mask in (train, validation, test))
display(pd.DataFrame(splits).T)
print(f'2025: {y.sum():,} labeled fraud cases out of {len(y):,} ({y.mean():.2%}).')

In [ ]:
monthly = y.groupby(dates.dt.month).agg(['size', 'sum', 'mean'])
fig, ax = plt.subplots(figsize=(8.2, 3.5))
ax.plot(monthly.index, monthly['mean'] * 100, color=teal, marker='o')
ax.set(xticks=range(1, 13), xlabel='Month in 2025', ylabel='Labeled fraud (%)',
       title='Fraud prevalence increased in December')
ax.grid(axis='y', alpha=.2)
show_chart(fig, 'monthly_fraud_rate.png')

eda = history.loc[train, ['valor_transacao', 'device_type']].assign(
    fraud=y[train], hour=dates[train].dt.hour)
hour_rates = eda.groupby('hour')['fraud'].agg(['size', 'sum', 'mean'])
device_rates = eda.groupby('device_type')['fraud'].agg(['size', 'sum', 'mean'])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(np.log1p(eda['valor_transacao']), bins=32, color=teal)
axes[0].set(xlabel='log(1 + amount in source units)', ylabel='Transactions', title='Training amounts')
axes[1].bar(hour_rates.index, hour_rates['mean'] * 100, color=teal)
axes[1].set(xlabel='Transaction hour', ylabel='Labeled fraud (%)', title='Training fraud by hour',
            xticks=range(0, 24, 3))
show_chart(fig, 'training_patterns.png')
display(eda.groupby('fraud')['valor_transacao'].quantile([.25, .5, .75, .95]).unstack())
display(device_rates)
print(f"Highest training fraud rate by hour: {hour_rates['mean'].idxmax():02d}:00 "
      f"({hour_rates['mean'].max():.2%}); hours with zero observed fraud: "
      f"{', '.join(str(h) for h in hour_rates.index[hour_rates['sum'].eq(0)])}.")

Hour has sharp changes in observed fraud rates. These may reflect the way this dataset was produced and require validation before using the pattern in another population. Amounts use the supplied units. The training summaries show associations.

## 2. Features and model comparison
Use log transaction amount, merchant type, city, device type, new-device status, hour and weekday. The log transform compresses the upper tail and preserves ordering. One-hot encoding lets each category, hour and weekday have its own effect. The encoder ignores unseen categories. Each model learns its preprocessing during fitting.

Exclude customer names and postal codes to limit memorization. Merchant name, region, amount category and time-of-day bin are omitted in favor of merchant type, city, amount and exact hour. Temperature has an unclear operational connection. The internal score has unclear provenance and little marginal association in the training audit.

The all-legitimate baseline shows the effect of imbalance. Logistic regression is a compact benchmark. Histogram gradient boosting can capture nonlinearities and interactions. Prefer logistic regression when its validation average precision is within 0.01 absolute of boosting.

In [ ]:
numeric = ['Log amount', 'New device']
categorical = ['Merchant type', 'City', 'Device type', 'Hour', 'Weekday']

def features(frame, dates):
    return pd.DataFrame({
        'Log amount': np.log1p(frame['valor_transacao']),
        'New device': frame['is_new_device'].astype(int),
        'Merchant type': frame['tipo_estabelecimento'],
        'City': frame['cidade'],
        'Device type': frame['device_type'],
        'Hour': dates.dt.hour,
        'Weekday': dates.dt.dayofweek,
    }, index=frame.index)

x = features(history, dates)
x_january = features(january, january_dates)
assert x.columns.tolist() == x_january.columns.tolist()
assert set(x.columns) == set(numeric + categorical) and target not in x
assert x_january.loc[0, 'Hour'] == january_dates.iloc[0].hour
assert np.isclose(x_january.loc[0, 'Log amount'], np.log1p(january.iloc[0]['valor_transacao']))

preprocess = ColumnTransformer([
    ('number', StandardScaler(), numeric),
    ('category', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
])
estimators = {
    'Logistic regression': LogisticRegression(max_iter=1000),
    'Histogram gradient boosting': HistGradientBoostingClassifier(
        max_iter=100, max_leaf_nodes=15, early_stopping=False, random_state=42),
}
models = {}
validation_scores = {}
for name, estimator in estimators.items():
    fitted = Pipeline([('preprocess', clone(preprocess)), ('model', estimator)])
    fitted.fit(x.loc[train], y[train])
    assert fitted.classes_.tolist() == [0, 1]
    models[name] = fitted
    validation_scores[name] = fitted.predict_proba(x.loc[validation])[:, 1]

def metrics(labels, scores, threshold):
    assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
    flags = scores >= threshold
    tn, fp, fn, tp = map(int, confusion_matrix(labels, flags, labels=[0, 1]).ravel())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn)
    return dict(tn=tn, fp=fp, fn=fn, tp=tp, precision=precision, recall=recall,
                f1=2*tp / (2*tp + fp + fn),
                false_positive_rate=fp / (fp + tn), alerts=tp + fp,
                alert_rate=float(flags.mean()), accuracy=(tp + tn) / len(labels),
                average_precision=float(average_precision_score(labels, scores)))

validation_metrics = [
    {'model': name, **metrics(y[validation], scores, .5)}
    for name, scores in validation_scores.items()
]
logistic, boosting = validation_metrics
selected_model = (logistic['model'] if logistic['average_precision'] + .01 >= boosting['average_precision']
                  else boosting['model'])
selected = models[selected_model]
display(pd.DataFrame(validation_metrics)[['model', 'average_precision', 'precision', 'recall', 'f1']])
print(f'Selected {selected_model}. Threshold-dependent metrics in this comparison use 0.5.')

## 3. Decision threshold
Average precision summarizes ranking across recall levels. F1 is the harmonic mean of precision and recall. The threshold maximizes F1 across distinct validation scores, with higher precision breaking ties. Remaining ties use the larger threshold. Review costs, capacity and customer impact should determine the eventual operating policy.

In [ ]:
val_scores = validation_scores[selected_model]
precision, recall, thresholds = precision_recall_curve(y[validation], val_scores)
# The final precision/recall pair represents an empty queue and has no threshold.
precision, recall = precision[:-1], recall[:-1]
total = precision + recall
f1 = np.divide(2 * precision * recall, total, out=np.zeros_like(total), where=total > 0)
best_f1 = np.isclose(f1, f1.max(), rtol=0, atol=1e-12)
best_precision = precision[best_f1].max()
best = np.flatnonzero(best_f1 & (precision == best_precision))[-1]
threshold = float(thresholds[best])
assert np.isclose(metrics(y[validation], val_scores, threshold)['f1'], f1.max())
assert metrics(y[validation], val_scores, threshold)['precision'] == best_precision

cutoffs = sorted({threshold, .5,
                  float(thresholds[np.argmin(abs(thresholds - threshold * .5))]),
                  float(thresholds[np.argmin(abs(thresholds - (1 + threshold) / 2))])})
threshold_tradeoffs = [
    {'threshold': cutoff, **metrics(y[validation], val_scores, cutoff)} for cutoff in cutoffs
]
display(pd.DataFrame(threshold_tradeoffs)[['threshold', 'precision', 'recall', 'f1', 'alerts', 'alert_rate']])
print(f'Frozen threshold: {threshold:.17g}')

## 4. December evaluation and error analysis
This uses the model trained on January–September with the threshold selected on October–November. Compare its ranking and review workload with an all-legitimate baseline. False-positive and false-negative rates below use legitimate and fraudulent transactions, respectively, as denominators.

In [ ]:
test_scores = selected.predict_proba(x.loc[test])[:, 1]
test_metrics = metrics(y[test], test_scores, threshold)
baseline_test_metrics = metrics(y[test], np.zeros(int(test.sum())), .5)
t = test_metrics
display(pd.DataFrame([{'approach': 'All legitimate', **baseline_test_metrics},
                      {'approach': selected_model, **t}]).set_index('approach').round(6).T)
display(pd.DataFrame([[t['tn'], t['fp']], [t['fn'], t['tp']]],
                     index=['Actual legitimate', 'Actual fraud'],
                     columns=['Predicted legitimate', 'Predicted fraud']))
display(Markdown(f"December precision was **{t['precision']:.2%}**, recall **{t['recall']:.2%}**, "
                 f"and average precision **{t['average_precision']:.4f}**. "
                 f"The queue contains **{t['alerts']:,}** alerts ({t['alert_rate']:.2%}), "
                 f"including **{t['fp']:,}** legitimate transactions. "
                 f"The model missed **{t['fn']:,}** labeled fraud cases."))
p, r, _ = precision_recall_curve(y[test], test_scores)
fig, ax = plt.subplots(figsize=(6.9, 4.0))
ax.step(r, p, where='post', color=teal, label=f"Model AP = {t['average_precision']:.3f}")
ax.axhline(y[test].mean(), color='#667482', ls='--', label='Constant-score baseline')
ax.scatter(t['recall'], t['precision'], color='#c2642f', zorder=3, label='Chosen threshold')
ax.set(xlabel='Recall', ylabel='Precision', title='December precision-recall curve', xlim=(0, 1.02), ylim=(0, 1.05))
ax.legend(loc='upper right', frameon=False)
show_chart(fig, 'precision_recall.png')

In [ ]:
actual = y[test]
flagged = pd.Series(test_scores >= threshold, index=actual.index)
errors = pd.DataFrame({
    'Amount band': pd.cut(history.loc[test, 'valor_transacao'], [-np.inf, 50, 100, 250, np.inf]),
    'Hour': dates[test].dt.hour,
    'Device': history.loc[test, 'device_type'],
    'Fraud': actual,
    'False positives': actual.eq(0) & flagged,
    'False negatives': actual.eq(1) & ~flagged,
})
error_analysis = {}
for column in ('Amount band', 'Hour', 'Device'):
    group = errors.groupby(column, observed=True).agg(
        Rows=('Fraud', 'size'), Fraud=('Fraud', 'sum'),
        FP=('False positives', 'sum'), FN=('False negatives', 'sum'))
    group['FPR'] = group['FP'] / (group['Rows'] - group['Fraud']).replace(0, np.nan)
    group['FNR'] = group['FN'] / group['Fraud'].replace(0, np.nan)
    assert group['FP'].sum() == t['fp'] and group['FN'].sum() == t['fn']
    display(Markdown(f'**December errors by {column.lower()}**'))
    display(group.round(4))
    group.index = group.index.map(str)
    error_analysis[column] = json.loads(group.reset_index().to_json(orient='records'))
    print(f"Most false positives: {group['FP'].idxmax()} ({group['FP'].max():,}). "
          f"Most false negatives: {group['FN'].idxmax()} ({group['FN'].max():,}).")

## 5. What the model relies on
Permutation importance shuffles one input column at a time and measures the decrease in validation average precision. A fixed random sample keeps the check small. Shuffling measures model reliance on this dataset. Correlation, feature interactions and unrealistic shuffled combinations can affect the result.

In [ ]:
sample = x.loc[validation].sample(n=8_000, random_state=42)
importance = permutation_importance(selected, sample, y.loc[sample.index],
                                    scoring='average_precision', n_repeats=3, random_state=42)
importance_table = pd.DataFrame({
    'feature': sample.columns,
    'mean_ap_drop': importance.importances_mean,
    'std_ap_drop': importance.importances_std,
}).sort_values('mean_ap_drop', ascending=False)
display(importance_table)
fig, ax = plt.subplots(figsize=(7.3, 3.9))
ordered = importance_table.iloc[::-1]
ax.barh(ordered['feature'], ordered['mean_ap_drop'], color=teal)
ax.set(xlabel='Mean decrease in average precision', title='Model reliance on a validation sample')
show_chart(fig, 'permutation_importance.png')
print('Largest measured AP decreases:', ', '.join(importance_table['feature'].head(3)))

## 6. Refit and classify January 2026
Refit the selected pipeline using all labeled 2025 transactions after evaluation. Reuse the full-precision validation threshold. The refit may change score distributions. December results describe the earlier evaluation model. January accuracy requires future labels.

The CSV preserves every January input field and row. `predicted_fraud` is 1 for scores at or above the cutoff and 0 otherwise. `fraud_score` is the positive-class model score. Its calibration has not been validated.

In [ ]:
final_model = clone(selected).fit(x, y)
january_scores = final_model.predict_proba(x_january)[:, 1]
assert np.isfinite(january_scores).all() and ((january_scores >= 0) & (january_scores <= 1)).all()
january_flags = (january_scores >= threshold).astype(int)
original = pd.read_csv(january_path, dtype=str, keep_default_na=False)
out = Path('submission/january_2026_predictions.csv')
original.assign(predicted_fraud=january_flags, fraud_score=january_scores).to_csv(
    out, index=False, float_format='%.17g')
saved = pd.read_csv(out, dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(saved.loc[:, original.columns], original)
saved_scores = np.array([float(value) for value in saved['fraud_score']])
assert saved.columns.tolist() == original.columns.tolist() + ['predicted_fraud', 'fraud_score']
assert len(saved) == 44_451 and np.array_equal(saved_scores, january_scores)
assert np.array_equal(saved['predicted_fraud'].astype(int), saved_scores >= threshold)
january_summary = dict(rows=len(saved), alerts=int(january_flags.sum()),
                       alert_rate=float(january_flags.mean()),
                       score_quantiles={str(q): float(np.quantile(january_scores, q))
                                        for q in (0, .25, .5, .75, .95, 1)})
display(Markdown(f"Scored **{len(saved):,}** January transactions and flagged "
                 f"**{january_summary['alerts']:,}** ({january_summary['alert_rate']:.2%}). "
                 "This is the predicted review share. Observed January fraud prevalence is unknown."))
daily = pd.Series(january_flags).groupby(january_dates.dt.day).sum()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.bar(daily.index, daily, color=teal)
ax.set(xlabel='Day in January 2026', ylabel='Review candidates', title='January alert volume by day')
show_chart(fig, 'january_daily_alerts.png')

results = dict(
    data=dict(historical_rows=len(y), historical_fraud_count=int(y.sum()), historical_fraud_rate=float(y.mean())),
    splits=splits, validation_metrics=validation_metrics, selected_model=selected_model,
    threshold=threshold, threshold_tradeoffs=threshold_tradeoffs, test_metrics=test_metrics,
    baseline_test_metrics=baseline_test_metrics, january=january_summary,
    feature_importance=importance_table.to_dict(orient='records'), error_analysis=error_analysis,
    versions=versions, runtime_seconds=round(perf_counter() - started, 2),
)
Path('_build/results.json').write_text(json.dumps(results, indent=2, allow_nan=False), encoding='utf-8')
print('January CSV verified against original fields, row order and the full-precision threshold.')

## 7. Recommendation and limits
Confirm the label maturation window and scoring-time feature availability. Agree on review capacity and the costs of errors. Run shadow scoring before a controlled review pilot. Track queue size and age, delayed-label precision and recall, drift, and customer friction. Review false negatives by amount and segment using denominators and sample sizes.

December prevalence and sharp hour patterns limit external validity. Feature availability and label quality remain assumptions. The January refit needs score and threshold monitoring. No January ground truth, intervention study, or measured savings are available.

References: [scikit-learn guidance on leakage](https://scikit-learn.org/1.8/common_pitfalls.html), [decision thresholds](https://scikit-learn.org/1.8/modules/classification_threshold.html), and [permutation importance](https://scikit-learn.org/1.8/modules/permutation_importance.html).